# **EDA Notebook**



---
## 0. Setup Environment

In [1]:
# Do not modify this code
%pip install uts_36106_core==0.0.3

from uts_36106_core.folders import *
from uts_36106_core.ipyrenders import *
import os

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.mount_gdrive()
at.create_folders()
if at.is_colab:
  os.chdir(f'{at.folder_path}')

import warnings
warnings.simplefilter(action='ignore')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT3/data


---
## Student Information

In [2]:
group_name = "Group 52"
student_name = "Andres Ethorimn Lobarb"
student_id = "26029165"

In [3]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [4]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [5]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [6]:
# No additional packages: pandas, numpy, scipy and altair are available on Colab.

### 0.b Import Packages

In [7]:
import altair as alt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr

In [8]:
# Utils

def column_profile(df):
    """One row per column: type, missing count and share, distinct values."""
    return pd.DataFrame({
        "type": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_share": df.isna().mean().round(4),
        "distinct": df.nunique(),
    })


def monthly_activity(transactions, timestamp_col, account_col, amount_col, intl_col):
    """Aggregate transactions per account and calendar month."""
    month = pd.to_datetime(transactions[timestamp_col]).dt.to_period("M")
    grouped = transactions.assign(month=month).groupby([account_col, "month"])
    return grouped.agg(
        monthly_spend=(amount_col, "sum"),
        transaction_count=(amount_col, "size"),
        international_share=(intl_col, "mean"),
    ).reset_index()


def forward_change(balances, account_col, date_col, balance_col, horizon):
    """Balance change from each month-end to the month-end `horizon` months later."""
    ordered = balances.sort_values([account_col, date_col])
    future = ordered.groupby(account_col)[balance_col].shift(-horizon)
    out = ordered[[account_col, date_col, balance_col]].copy()
    out["future_change"] = future - ordered[balance_col]
    out["month"] = pd.to_datetime(out[date_col]).dt.to_period("M")
    return out


def spearman_summary(x, y, label):
    """Spearman correlation with a Fisher-z 95% interval and a size label."""
    mask = x.notna() & y.notna()
    rho, p_value = spearmanr(x[mask], y[mask])
    n = int(mask.sum())
    half_width = 1.96 / np.sqrt(n - 3)
    low, high = np.tanh(np.arctanh(rho) - half_width), np.tanh(np.arctanh(rho) + half_width)
    size = abs(rho)
    magnitude = "negligible" if size < 0.1 else "small" if size < 0.3 else "medium" if size < 0.5 else "large"
    return pd.Series({"feature": label, "n": n, "rho": round(rho, 4), "ci_low": round(low, 4),
                      "ci_high": round(high, 4), "p_value": p_value, "magnitude": magnitude})


def binned_mean(df, feature, target, n_bins):
    """Mean of `target` with a 95% interval per quantile bin of `feature`."""
    bins = pd.qcut(df[feature].rank(method="first"), n_bins, labels=False) + 1
    grouped = df.assign(bin=bins).groupby("bin")
    table = grouped.agg(feature_median=(feature, "median"), mean_target=(target, "mean"),
                        sd=(target, "std"), n=(target, "size")).reset_index()
    table["ci_half"] = 1.96 * table["sd"] / np.sqrt(table["n"])
    table["low"], table["high"] = table["mean_target"] - table["ci_half"], table["mean_target"] + table["ci_half"]
    return table


def binned_chart(table, x_title, y_title):
    """Point-and-interval chart of a binned mean, with a zero reference line."""
    base = alt.Chart(table).encode(x=alt.X("bin:O", title=x_title))
    band = base.mark_rule(color="#898781").encode(y=alt.Y("low:Q", title=y_title), y2="high:Q")
    points = base.mark_point(filled=True, color="#2a78d6").encode(
        y="mean_target:Q", tooltip=["bin", "feature_median", "mean_target", "low", "high", "n"])
    zero = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(color="#e34948", strokeDash=[4, 4]).encode(y="y:Q")
    return (band + points + zero).properties(width=520, height=260)

In [9]:
# Constants
# Provisional working horizon (months ahead). It is a working assumption until the balance
# history EDA has examined other horizons; it is not a decision taken here.
HORIZON = 3
N_BINS = 10

---
## B. Data Understanding

**Approach.** This notebook studies the `transactions` table on its own terms first, then asks one question per feature: does it relate to how an account's balance moves over the following months? Each finding is stated only from the output directly above it. Decisions that need several tables (the forecast horizon, which accounts to forecast, the business use) are not taken here; they are taken after all three EDA notebooks, from the evidence they produce.

In [11]:
# Do not modify this code
try:
  df = pd.read_csv(at.folder_path / "transactions.csv")
except Exception as e:
  print(e)

### B.1 Explore Dataset

In [12]:
print("Rows, columns:", df.shape)
print("Duplicate transaction_id:", df["transaction_id"].duplicated().sum())
print("Fully duplicated rows (ignoring the id):", df.drop(columns="transaction_id").duplicated().sum())
timestamps = pd.to_datetime(df["transaction_timestamp"])
print("First and last timestamp:", timestamps.min(), timestamps.max())
print("Accounts:", df["account_id"].nunique(), "| customers:", df["customer_id"].nunique())
display(column_profile(df))

# Is there a direction (inflow vs outflow) in the amounts?
amount_by_type = df.groupby("transaction_type")["amount"].describe(percentiles=[0.25, 0.5, 0.75])
print("Amount by transaction type (dollars):")
display(amount_by_type.round(2))
print("Transactions with a zero or negative amount:", (df["amount"] <= 0).sum())

print("Transaction type by channel (counts):")
display(pd.crosstab(df["transaction_type"], df["channel"]))

Rows, columns: (1532009, 24)
Duplicate transaction_id: 0
Fully duplicated rows (ignoring the id): 0
First and last timestamp: 2023-01-01 00:00:00 2025-12-30 23:59:00
Accounts: 38562 | customers: 15000


,type,missing,missing_share,distinct
transaction_id,int64,0,0.0000,1532009
customer_id,int64,0,0.0000,15000
account_id,int64,0,0.0000,38562
card_id,float64,390141,0.2547,33555
merchant_id,int64,0,0.0000,150
transaction_timestamp,object,0,0.0000,979531
amount,float64,0,0.0000,151968
amount_band,object,0,0.0000,5
transaction_type,object,0,0.0000,5
channel,object,0,0.0000,4


Amount by transaction type (dollars):


,count,mean,std,min,25%,50%,75%,max
transaction_type,,,,,,,,
ATM Withdrawal,306499.0,340.19,372.01,6.15,108.10,224.61,434.06,8987.12
Bill Payment,306029.0,340.09,372.43,7.02,107.62,224.05,434.20,9318.58
Direct Debit,307418.0,340.14,372.22,7.16,107.99,224.35,433.63,7799.19
Purchase,305518.0,339.37,369.84,7.16,108.25,224.78,433.64,7378.10
Transfer,306545.0,339.39,372.26,3.60,107.46,224.21,432.56,6881.69


Transactions with a zero or negative amount: 0
Transaction type by channel (counts):


channel,ATM,Mobile,POS,Web
transaction_type,,,,
ATM Withdrawal,15204,61231,183969,46095
Bill Payment,15382,61004,183668,45975
Direct Debit,15448,61526,183933,46511
Purchase,15206,61072,183564,45676
Transfer,15407,61299,183928,45911


In [13]:
monthly_counts = (timestamps.dt.to_period("M").astype(str).value_counts().sort_index()
                  .rename_axis("month").reset_index(name="transactions"))
alt.Chart(monthly_counts).mark_bar(color="#2a78d6").encode(
    x=alt.X("month:O", title="Month"),
    y=alt.Y("transactions:Q", title="Transactions"),
    tooltip=["month", "transactions"],
).properties(width=620, height=240)

alt.Chart(...)

In [14]:
dataset_insights = """
Size and coverage. The table holds 1532009 transactions and 24 columns for 38562 accounts
of 15000 customers, from 2023-01-01 to 2025-12-30. Every one of the 36 months has between
roughly 39000 and 44000 transactions (chart), so there is no gap or partial month to repair.

Quality. There are no duplicate transaction ids and no duplicated rows. The only missing
field is <card_id> (390141 rows, 25.5%); the cause is not checked here. The file carries more
columns than the warehouse schema (<amount_band>, merchant name, location and risk fields,
currency, <customer_present_flag>, <aml_risk>).

Direction of money. Every amount is positive (minimum $3.60; no zero or negative amount) for
all five transaction types, Transfer included. The table therefore records how much activity
an account has, not whether money came in or went out: a net flow per account cannot be
computed from it.

Type and channel. The five types have almost the same amount distribution (mean about $340,
median about $224 in each), and every type appears through every channel in almost the same
proportions; for example 183969 ATM withdrawals are recorded through POS. Type and channel
therefore carry little descriptive information and are not studied further as features.

What is tested next. Three account-month features that this table can support: total monthly
spend (B.2), number of transactions in the month (B.3) and the share of international
transactions (B.4), each against the balance change over the following months.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='dataset_insights', value=dataset_insights)

### B.2 Explore Feature of Interest `monthly_spend`

In [ ]:
# Complementary load: balances are needed to measure the change that follows each month.
balances = pd.read_csv(at.folder_path / "account_balance_history.csv",
                       usecols=["account_id", "balance_date", "balance"])
changes = forward_change(balances, "account_id", "balance_date", "balance", HORIZON)

activity = monthly_activity(df, "transaction_timestamp", "account_id", "amount", "international_transaction_flag")
first_month = activity["month"].min()

# One row per account and month-end from the first transaction month; months with no
# transactions get zero spend and zero count (no transaction recorded that month).
panel = changes[changes["month"] >= first_month].merge(activity, on=["account_id", "month"], how="left")
panel[["monthly_spend", "transaction_count"]] = panel[["monthly_spend", "transaction_count"]].fillna(0)
panel = panel.dropna(subset=["future_change"])
print("Account-months with an observed future change:", len(panel))
print("Share of account-months with no transaction:", round((panel["transaction_count"] == 0).mean(), 4))
display(panel["monthly_spend"].describe().round(2).to_frame("monthly spend (dollars)"))

spend_test = spearman_summary(panel["monthly_spend"], panel["future_change"], "monthly spend")
display(spend_test.to_frame("Spearman, monthly spend vs balance change over the next HORIZON months"))

In [ ]:
spend_bins = binned_mean(panel, "monthly_spend", "future_change", N_BINS)
binned_chart(spend_bins, "Monthly spend decile (1 = lowest)", f"Mean balance change over the next {HORIZON} months ($)")

In [ ]:
feature_1_insights = """
Feature. Total amount of an account's transactions in a calendar month, joined to the change
in its balance over the following <HORIZON> month-ends (3, a provisional working horizon).
There are 1272546 account-months with an observed future change from January 2023; 37.9% of
them have no transaction. Spend is highly skewed (median $150, mean $375, maximum $9588).

Test choice. The skew and the mass of zeros rule out a test that assumes normality, and the
question is whether more spend goes with a larger or smaller later change, so the Spearman
rank correlation answers it, with its value as the effect size.

Result. rho = 0.0016 (95% interval -0.0001 to 0.0034, p = 0.063): negligible. The chart shows
the mean change over the next 3 months between about $640 and $720 in every spend decile, with
overlapping intervals and no trend. Monthly spend does not order accounts by how their
balance moves next.

Reading. The mean change is positive in every decile; that is a general upward movement of
balances in this period, not an effect of spend. Limitation: only one horizon is examined here;
other horizons are examined with the balance history itself.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### B.3 Explore Feature of Interest `transaction_count`

In [ ]:
display(panel["transaction_count"].value_counts().sort_index().to_frame("account-months"))
count_test = spearman_summary(panel["transaction_count"], panel["future_change"], "transaction count")
display(count_test.to_frame("Spearman, transactions in the month vs balance change over the next HORIZON months"))

In [ ]:
count_table = (panel.assign(transactions=panel["transaction_count"].clip(upper=5))
               .groupby("transactions")["future_change"].agg(["mean", "std", "size"]).reset_index())
count_table["low"] = count_table["mean"] - 1.96 * count_table["std"] / np.sqrt(count_table["size"])
count_table["high"] = count_table["mean"] + 1.96 * count_table["std"] / np.sqrt(count_table["size"])
base = alt.Chart(count_table).encode(x=alt.X("transactions:O", title="Transactions in the month (5 = five or more)"))
(base.mark_rule(color="#898781").encode(y=alt.Y("low:Q", title=f"Mean balance change over the next {HORIZON} months ($)"), y2="high:Q")
 + base.mark_point(filled=True, color="#2a78d6").encode(y="mean:Q", tooltip=["transactions", "mean", "low", "high", "size"])
 + alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(color="#e34948", strokeDash=[4, 4]).encode(y="y:Q")
).properties(width=520, height=260)

In [ ]:
feature_2_insights = """
Feature. Number of transactions an account makes in a calendar month (0 to 11; 482064
account-months with none, 405407 with one, 1044 with seven or more).

Test choice. A count with a long right tail; the same rank question as B.2, so Spearman.

Result. rho = 0.0012 (95% interval -0.0006 to 0.0029, p = 0.19): negligible. In the chart the
mean change over the next 3 months stays near $680 to $740 for every count, and the wider
intervals at four and five or more transactions reflect fewer account-months, not a trend.
Account activity in a month does not indicate how the balance moves next.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### B.4 Explore Feature of Interest `international_share`

In [ ]:
# Only months with at least one transaction have an international share.
with_activity = panel[panel["transaction_count"] > 0]
print("Account-months with at least one transaction:", len(with_activity))
display(with_activity["international_share"].value_counts(bins=[-0.01, 0, 0.25, 0.5, 0.75, 1]).sort_index()
        .to_frame("account-months"))
intl_test = spearman_summary(with_activity["international_share"], with_activity["future_change"], "international share")
display(intl_test.to_frame("Spearman, international share vs balance change over the next HORIZON months"))

In [ ]:
intl_table = (with_activity.assign(share=pd.cut(with_activity["international_share"], [-0.01, 0, 0.5, 1],
                                                labels=["none", "up to half", "more than half"]))
              .groupby("share", observed=True)["future_change"].agg(["mean", "std", "size"]).reset_index())
intl_table["low"] = intl_table["mean"] - 1.96 * intl_table["std"] / np.sqrt(intl_table["size"])
intl_table["high"] = intl_table["mean"] + 1.96 * intl_table["std"] / np.sqrt(intl_table["size"])
base = alt.Chart(intl_table).encode(x=alt.X("share:O", title="Share of the month's transactions that are international", sort=None))
(base.mark_rule(color="#898781").encode(y=alt.Y("low:Q", title=f"Mean balance change over the next {HORIZON} months ($)"), y2="high:Q")
 + base.mark_point(filled=True, color="#2a78d6").encode(y="mean:Q", tooltip=["share", "mean", "low", "high", "size"])
 + alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(color="#e34948", strokeDash=[4, 4]).encode(y="y:Q")
).properties(width=420, height=260)

In [ ]:
feature_n_insights = """
Feature. Share of an account's transactions in a month made abroad, defined only for the
790482 account-months with at least one transaction. Most have no international transaction
(675507); because accounts make few transactions, the share takes few values (for example 0.5
or 1).

Test choice. Bounded and lumpy values; the rank question again, so Spearman.

Result. rho = 0.0001 (95% interval -0.0021 to 0.0023, p = 0.94): negligible. The chart shows the
same mean change (about $690) for none, up to half and more than half international.

Summary of this notebook. None of the three features this table supports shows a relation
with the balance change over the next 3 months beyond a negligible size (summary table above).
On this evidence, transaction behaviour as recorded here is not a basis for forecasting the
balance; whether the balance history itself is, is examined in its own notebook.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_n_insights', value=feature_n_insights)

In [ ]:
# Summary of the three feature tests in this notebook
display(pd.DataFrame([spend_test, count_test, intl_test]).set_index("feature"))

## C. Ethical Consideration of Features


In [ ]:
# Fields that can act as proxies for protected or sensitive traits
sensitive = ["merchant_category", "merchant_country", "currency_code", "merchant_state"]
for column in sensitive:
    print(column, "->", df[column].value_counts(normalize=True).round(3).head(6).to_dict())

In [ ]:
ethical_considerations = """
Several fields describe a customer's life more than their banking. Merchant category reveals
health spending (Healthcare, 8.0% of transactions) and the high-risk category; merchant
country and currency (8.7% outside Australia) can stand in for nationality or migrant status;
merchant state locates where people spend. A model using them could treat customers
differently because of health, origin or place, which the bank must not do.

Action. Only account-level monthly summaries (spend, count, international share) are carried
forward, and only if a later notebook shows they add evidence. If the international share is
ever used, forecast errors are audited by customer group before any use, and the result is a
prompt for staff review, never an automatic decision on an account.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='ethical_considerations', value=ethical_considerations)